In [1]:
from func_parallel import model as new_model
from func_parallel import model_setup
from dask.distributed import Client, LocalCluster
import dask
from dask import delayed, compute
import numpy as np
import xarray as xr
import time as tm

# ------------------------
# Start Dask cluster
# ------------------------
try:
    client.close()
    cluster.close()
except NameError:
    pass

cluster = LocalCluster(
    n_workers=4,           # CPU-bound optimal for M1
    threads_per_worker=1,  # safer for xsimlab
    processes=True,        # separate interpreters → avoids context clashes
    memory_limit='3GB',
)
client = Client(cluster)

# ------------------------
# Parameters
# ------------------------
param_name = 'HigherOrderMortality__rate'
param_values = np.linspace(0.01, 0.02, 100)

# Batch parameters to reduce Dask overhead
batch_size = 10
batches = [param_values[i:i+batch_size] for i in range(0, len(param_values), batch_size)]

# ------------------------
# Delayed batch run
# ------------------------
@delayed
def run_batch(param_list):
    batch_datasets = []
    for r in param_list:
        ds = model_setup.xsimlab.update_vars(input_vars={param_name: r}, model=new_model).xsimlab.run(model=new_model)
        ds['time'] = ds.time.round(9)
        ds = ds.assign_coords({param_name: r}).expand_dims(param_name)
        batch_datasets.append(ds)
    # Combine batch internally
    return xr.combine_by_coords(batch_datasets)

# ------------------------
# Build delayed tasks
# ------------------------
tasks = [run_batch(batch) for batch in batches]

# ------------------------
# Trigger computation and combine all batches at once
# ------------------------
start = tm.time()
# Compute all batches in parallel
batch_results = compute(*tasks)
# Combine batches into a single dataset
final_result = xr.combine_by_coords(batch_results)
end = tm.time()

print("Total runtime:", round(end-start, 5), "seconds")
print(final_result)


Total runtime: 163.93787 seconds
<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOr

2025-10-18 00:14:11,444 - tornado.application - ERROR - Uncaught exception GET /system/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='localhost:8787', method='GET', uri='/system/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/websocket.py", line 938, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/web.py", line 3301, in wrapper
    return method(self, *args, **kwargs)
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if

In [1]:
from func_parallel import model as new_model
from func_parallel import model_setup
from dask.distributed import Client, LocalCluster
import dask
from dask import delayed
import numpy as np
import xarray as xr

# ------------------------
# Start Dask cluster
# ------------------------
try:
    client.close()
    cluster.close()
except:
    pass

cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=2,
    processes=True,
    memory_limit='3GB',
)
client = Client(cluster)

# ------------------------
# Delayed model run
# ------------------------

@delayed
def run_model_test(param_dict):
    model_out = model_setup.xsimlab.update_vars(input_vars=param_dict, model=new_model).xsimlab.run(model=new_model)
    #model_out = model_setup_updated.xsimlab.run(model=new_model)
    model_out['time'] = model_out.time.round(9)
    return model_out

# ------------------------
# Wrap delayed dataset with assigned coordinate
# ------------------------
@delayed
def add_param_coord(ds, param_name, param_val):
    # assign the varying parameter as coordinate
    # expand along this new dimension
    return ds.assign_coords({param_name: param_val}).expand_dims(param_name)

# ------------------------
# Build tasks
# ------------------------
param_name = 'HigherOrderMortality__rate'
param_values = np.linspace(0.01, 0.02, 100)
tasks = [run_model_test({param_name: r}) for r in param_values]

lazy_datasets = [add_param_coord(task, param_name, r) for task, r in zip(tasks, param_values)]

# ------------------------
# Combine by coordinates lazily
# ------------------------
@delayed
def combine_lazy(datasets):
    return xr.combine_by_coords(datasets)

combined_delayed = combine_lazy(lazy_datasets)

# ------------------------
# Trigger computation once
# ------------------------
import time as tm
start = tm.time()
final_result = combined_delayed.compute()
end = tm.time()
print(round(end-start,5),"seconds")
print(final_result)

167.00046 seconds
<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOrderMortality__r

In [1]:
from func_parallel import model as new_model
from func_parallel import model_setup
from dask.distributed import Client, LocalCluster
import dask
from dask import delayed
import numpy as np
import xarray as xr

# ------------------------
# Start Dask cluster
# ------------------------
try:
    client.close()
    cluster.close()
except:
    pass

cluster = LocalCluster(
    n_workers=6,
    threads_per_worker=2,
    processes=True,
    memory_limit='3GB',
)
client = Client(cluster)

# ------------------------
# Define delayed model run
# ------------------------
@delayed
def run_model_test(param_dict):
    model_setup_updated = model_setup.xsimlab.update_vars(input_vars=param_dict, model=new_model)
    model_out = model_setup_updated.xsimlab.run(model=new_model)
    model_out['time'] = model_out.time.round(9)
    return model_out

# ------------------------
# Build tasks for parameter sweep
# ------------------------
param_values = np.linspace(0.01, 0.02, 100)
param_name = 'HigherOrderMortality__rate'
tasks = [run_model_test({param_name: r}) for r in param_values]

# ------------------------
# Wrap delayed datasets to add parameter coordinate
# ------------------------
@delayed
def add_param_coord(ds, param_val):
    return ds.expand_dims({param_name: [param_val]})

lazy_datasets = [add_param_coord(task, r) for task, r in zip(tasks, param_values)]

# ------------------------
# Combine by coordinates lazily
# ------------------------
@delayed
def combine_lazy(datasets):
    return xr.combine_by_coords(datasets)

combined_delayed = combine_lazy(lazy_datasets)

# ------------------------
# Trigger computation once
# ------------------------
import time as tm
start = tm.time()
final_result = combined_delayed.compute()
end = tm.time()
print(round(end-start,5),"seconds")
print(final_result)


101.8401 seconds
<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOrderMortality__ra

In [2]:
final_result

<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOrderMortality__rate) float64 800B ...
    GGE__alpha                                      (HigherOrderMortality__rate) float64 800B ...
    GGE__assimilated_consumer                       (HigherOrderMortality__rate) <U1 400B ...
    GGE__assimilation_value                         (HigherOrderMortality__rate, zoo, time) float64 16MB ...
    GGE__egested_detritus                           (HigherOrderMortality__rate) <U1 400B ...
    ...                                              ...
    Phytoplankton__phyto_index                      (HigherOrderMortality__rate, phyto) float64 2kB ...
    Time__time_input                                (HigherOrderMortality__rate, time) int64 4MB ...
    Zooplankton__biomass                            (HigherOrderMortality__rate, zoo, time) float64 16MB ...
    Zooplankton__biomass_init                       (HigherOrderMortality__rate, zoo) float64 3kB ...
    Zooplankton__biomass_label                      (HigherOrderMortality__rate) <U1 400B ...
    Zooplankton__zoo_index                          (HigherOrderMortality__rate, zoo) float64 3kB ...

In [2]:
from func_parallel import model as new_model
from func_parallel import model_setup
from dask.distributed import Client, LocalCluster
import dask
from dask import delayed
import numpy as np
import xarray as xr

# ------------------------
# Start Dask cluster
# ------------------------
try:
    client.close()
    cluster.close()
except:
    pass

cluster = LocalCluster(
    n_workers=6,
    threads_per_worker=2,
    processes=True,
    memory_limit='3GB',
)
client = Client(cluster)

# ------------------------
# Define delayed model run
# ------------------------
@delayed
def run_model_test(param_dict):
    model_setup_updated = model_setup.xsimlab.update_vars(input_vars=param_dict, model=new_model)
    model_out = model_setup_updated.xsimlab.run(model=new_model)
    model_out['time'] = model_out.time.round(9)
    return model_out

# ------------------------
# Build tasks for parameter sweep
# ------------------------
param_values = np.linspace(0.01, 0.02, 100)
tasks = [run_model_test({'HigherOrderMortality': {'rate': r}}) for r in param_values]

# ------------------------
# Wrap each delayed Dataset to add parameter coordinate
# ------------------------
def add_param_coord(ds, param_val):
    ds = ds.copy()
    ds = ds.expand_dims({'HigherOrderMortality__rate':[param_val]})
    return ds

lazy_datasets = [delayed(add_param_coord)(task, r) for task, r in zip(tasks, param_values)]

# ------------------------
# Concatenate all delayed datasets lazily
# ------------------------
combined_delayed = delayed(xr.concat)(lazy_datasets, dim='HigherOrderMortality__rate')

# ------------------------
# Trigger computation once
# ------------------------
final_result = combined_delayed.compute()  # executes all tasks at once

print(final_result)


<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOrderMortality__rate) float64 800B 

In [1]:
from func_parallel import model as new_model
from func_parallel import model_setup#, run_model_test

In [2]:
from dask.distributed import Client, LocalCluster


try:
    client.close()
    cluster.close()
except:
    pass

    
cluster = LocalCluster(
    n_workers=6,           # good balance on M1 — test 4, 6, 8
    threads_per_worker=2,  # crucial for xsimlab (no shared context)
    processes=True,        # separate interpreters → no context clash
    memory_limit='3GB',    # prevents macOS swap thrashing
)
client = Client(cluster)
#client

2025-10-17 13:47:28,493 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='localhost:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/websocket.py", line 938, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/web.py", line 3301, in wrapper
    return method(self, *args, **kwargs)
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if

In [9]:
import numpy as np
import dask
from dask import delayed, compute
import xarray as xr

@delayed
def run_model_test(i):
    model_setup_updated = model_setup.xsimlab.update_vars(input_vars=i,model=new_model)
    model_out = model_setup_updated.xsimlab.run(model=new_model)
    
    # solve ivp introduces rounding errors for time, which mess up combining datasets later: hence rounding
    model_out['time']= model_out.time.round(9)
    return model_out

# Build delayed tasks
param_values=np.linspace(0.01, 0.02, 100)
tasks = [run_model_test({'HigherOrderMortality': {'rate': r}}) for r in param_values]


In [14]:
# Turn delayed objects into actual (lazy) datasets
lazy_datasets = dask.compute(*tasks, sync=False)

combined = xr.concat(lazy_datasets, dim='HigherOrderMortality__rate')
combined = combined.assign_coords(param=('HigherOrderMortality__rate', param_values))

final_result = combined.compute()  # triggers distributed execution

TypeError: can only concatenate xarray Dataset and DataArray objects, got <class 'distributed.client.Future'>

In [15]:
# tasks are delayed xarray.Dataset objects
lazy_datasets = tasks  # do NOT compute yet

# add parameter coordinate using delayed mapping
import dask
import xarray as xr

def add_param_coord(ds, val):
    return ds.assign_coords(HigherOrderMortality__rate=val)

lazy_datasets_with_coord = [dask.delayed(add_param_coord)(ds, val) 
                            for ds, val in zip(lazy_datasets, param_values)]

# Use xr.concat with Dask arrays (will stay lazy)
combined = xr.concat([xr.DataArray(ds) if isinstance(ds, xr.Dataset) else ds 
                      for ds in lazy_datasets_with_coord], dim='HigherOrderMortality__rate')

# Compute once
final_result = combined.compute()


TypeError: can only concatenate xarray Dataset and DataArray objects, got <class 'dask.delayed.Delayed'>

In [12]:
final_result

<xarray.Dataset> Size: 252MB
Dimensions:                                         (
                                                     HigherOrderMortality__rate: 100,
                                                     zoo: 4, time: 5000,
                                                     phyto: 3, full: 7, clock: 2)
Coordinates:
  * HigherOrderMortality__rate                      (HigherOrderMortality__rate) float64 800B ...
  * clock                                           (clock) int64 16B 0 1
  * phyto                                           (phyto) float64 24B 0.63 ...
  * time                                            (time) float64 40kB 0.0 ....
  * zoo                                             (zoo) float64 32B 6.3 ......
    param                                           (HigherOrderMortality__rate) float64 800B ...
Dimensions without coordinates: full
Data variables: (12/56)
    Core__solver_type                               (HigherOrderMortality__rate) <U9 4kB ...
    GGE__R                                          (HigherOrderMortality__rate) float64 800B ...
    GGE__alpha                                      (HigherOrderMortality__rate) float64 800B ...
    GGE__assimilated_consumer                       (HigherOrderMortality__rate) <U1 400B ...
    GGE__assimilation_value                         (HigherOrderMortality__rate, zoo, time) float64 16MB ...
    GGE__egested_detritus                           (HigherOrderMortality__rate) <U1 400B ...
    ...                                              ...
    Phytoplankton__phyto_index                      (HigherOrderMortality__rate, phyto) float64 2kB ...
    Time__time_input                                (HigherOrderMortality__rate, time) int64 4MB ...
    Zooplankton__biomass                            (HigherOrderMortality__rate, zoo, time) float64 16MB ...
    Zooplankton__biomass_init                       (HigherOrderMortality__rate, zoo) float64 3kB ...
    Zooplankton__biomass_label                      (HigherOrderMortality__rate) <U1 400B ...
    Zooplankton__zoo_index                          (HigherOrderMortality__rate, zoo) float64 3kB ...

In [3]:
import numpy as np
import dask
from dask import delayed, compute

#@delayed
#def run_model_with_params(param_dict):
#    out = model_setup.xsimlab.run(model=new_model)
#    return out


# Compute in parallel
#results = client.compute(tasks, scheduler='processes')


In [22]:
from dask.distributed import progress

import time as tm
start = tm.time()

futures = client.compute(tasks)
#progress(futures, notebook=False)

results = client.gather(futures)

end = tm.time()

print(round(end-start,5),"seconds")

2025-10-17 13:38:36,383 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:63399 -> None
Traceback (most recent call last):
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/iostream.py", line 962, in _handle_write
    num_bytes = self.write_to_fd(self._write_buffer.peek(size))
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/tornado/iostream.py", line 1124, in write_to_fd
    return self.socket.send(data)  # type: ignore
           ~~~~~~~~~~~~~~~~^^^^^^
OSError: [Errno 55] No buffer space available

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/distributed/worker.py", line 1797, in get_data
    response = await comm.read(deserializers=serializers)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/

63.88576 seconds


In [12]:
import psutil
psutil.cpu_count(logical=False)   # physical cores
psutil.cpu_count(logical=True)    # total logical cores


8

In [7]:
import time as tm
start = tm.time()
run_model_with_params({'HigherOrderMortality': {'rate': 0.01}})
end = tm.time()
print(round(end-start,5),"seconds")

0.00033 seconds


In [8]:
from multiprocessing import Pool

def generate_iterable_parscan(parameter, par_range):
    return [{parameter:val} for val in par_range]

iter_scan = generate_iterable_parscan('HigherOrderMortality__rate', np.linspace(0.01, 0.02, 100))

#run_model_test(iter_scan[2]).Phytoplankton__biomass.plot.line(x='time')

import time as tm
p = Pool(processes=20)
start = tm.time()
data = p.map(run_model_test, iter_scan)
end = tm.time()
p.close()
print(round(end-start,5),"seconds")

69.1116 seconds


In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# import necessary packages
import numpy as np
import matplotlib.pyplot as plt
import xso

from Stocketal2008_comps import (Nutrient, PhytoSizeSpectrum, ZooSizeSpectrum, 
    ConstantExternalNutrient, LinearForcingInput, 
    MonodGrowth_SizeBased, 
    StockGrazingMatrix, Stock_GGE_MatrixGrazing, 
    StockPhytoMortality, StockZooMortality, StockZooMortality_simpleinput)



# number size classes of phytoplankton and zooplankton
P_num = 3
Z_num = 4

# create initial biomass
phyto_init = np.tile(1.5, (P_num))
zoo_init = np.tile(.1, (Z_num))

# calculate log-spaced size classes from ranges and total number
phyto_sizes = [0.63, 6.3, 63]
zoo_sizes = [6.3, 63, 630, 6300]

# Phytoplankton parameters
phyto_ks = [0.062, 0.45, 3.3]
phyto_mu0 = [1.0, 1.26, 0.42]
phyto_mortality = [1.0, 0, 0]
phyto_mort_exponent = [2.0, 1.0, 1.0]
phyto_recycling = [1.0, 0.0, 0.0]

# Zooplankton parameters
zoo_imax = [10.0, 3.3, 1.1, 0.36]
zoo_Ki = 3.0
zoo_frac_assim = 0.7  # alpha
zoo_frac_excreted = 0.45  # R
zoo_gge = 0.25
# prey availability: basically just 1 for size class below of Z and P, no other grazing! issa matrix 4 x 7
zoo_prey_avail = np.array([[1, 0, 0, 0, 0, 0, 0], # Z1
                          [0, 1, 0, 1, 0, 0, 0], # Z2
                          [0, 0, 1, 0, 1, 0, 0], # Z3
                          [0, 0, 0, 0, 0, 1, 0]]) # Z4

# dens dep prey exploitation factor
zoo_frac_egest_recycled = [1, 1, 0, 0]

# zoo mortality
zoo_higherordermortality = 0.0093
zoo_mortality_array = [0, 0, 0, zoo_higherordermortality]
zoo_mort_exponent = [0, 0, 0, 1]
zoo_frac_mortylity_recycled = [0, 0, 0, 0.5]



nutrient_input = 0.0053 # 0.017



model = xso.create({
            # State variables
            'Nutrient': Nutrient,
            'Phytoplankton': PhytoSizeSpectrum,
            'Zooplankton': ZooSizeSpectrum,
        
            # Flows:
            'Inflow': LinearForcingInput,
        
            # Growth
            'Growth': MonodGrowth_SizeBased,
        
            # Grazing
            'Grazing': StockGrazingMatrix,
            'GGE': Stock_GGE_MatrixGrazing,
        
            # Mortality
            'PhytoMortality': StockPhytoMortality,
            'HigherOrderMortality': StockZooMortality_simpleinput,
        
            # Forcings
            'N0': ConstantExternalNutrient,
        })
        
model_setup = xso.setup(solver='solve_ivp', model=model,
        time=np.arange(0,5000),
        input_vars={
                # State variables
                'Nutrient':{'value_label':'N','value_init':1.0},
                'Phytoplankton':{'biomass_label':'P','biomass_init':phyto_init, 'phyto_index':phyto_sizes},
             
                'Zooplankton':{'biomass_label':'Z','biomass_init':zoo_init, 'zoo_index': zoo_sizes},
            
                # Flows:
                'Inflow':{'forcing':'N0', 'rate':1., 'var':'N'},
            
                # Growth
                'Growth':{'resource':'N', 'consumer':'P', 'halfsat':phyto_ks, 'mu_max':phyto_mu0},

                # Grazing
                'Grazing':{'resource':'P', 'consumer':'Z', 'Imax':zoo_imax, 'KsZ':zoo_Ki, 'phiPZ':zoo_prey_avail},
                'GGE':{'grazed_phyto':'P', 'grazed_zoo':'Z', 'assimilated_consumer':'Z', 'egested_detritus':'N', 
                       'R':zoo_frac_excreted, 'alpha':zoo_frac_assim, 'f_I':zoo_frac_egest_recycled, 'gge':zoo_gge},
            
                # Mortality
                'PhytoMortality':{'population':'P', 'nutrient':'N', 'rate':phyto_mortality, 'exponent':phyto_mort_exponent, 'recycling':phyto_recycling},
                'HigherOrderMortality':{'population':'Z', 'nutrient':'N', 'rate':zoo_higherordermortality, 'exponent':zoo_mort_exponent, 'recycling':zoo_frac_mortylity_recycled},
 # Forcings
                'N0':{'forcing_label':'N0', 'value':nutrient_input},
        })


(<class 'xsimlab.process.Backend'>, <class 'xso.backendcomps.Backend'>, <class 'object'>)
<class 'xsimlab.process.Backend'>
xsimlab.process


In [2]:
from func_parallel_tests import run_model_test_XX

In [2]:
model.Core.__class__.__mro__


(xsimlab.process.Backend, xso.backendcomps.Backend, object)

In [3]:
print(model.Core.__class__)
print(model.Core.__module__)


<class 'xsimlab.process.Backend'>
xso.backendcomps


In [1]:
def generate_iterable_parscan(parameter, par_range):
    return [{parameter:val} for val in par_range]

In [6]:
from functools import partial
from multiprocessing import Pool
import time as tm
import numpy as np

# model and model_setup created as before



if __name__ == '__main__':

        # Create the partial function
    run_partial = partial(run_model_test_XX, model_setup=model_setup, model=model)
    
    # Define the iterable for your parameter scan
    iter_scan = generate_iterable_parscan('N0__value', np.linspace(0.001, 0.2, 5))


    # Run in parallel
    p = Pool(processes=4)
    start = tm.time()
    data = p.map(run_partial, iter_scan)
    end = tm.time()
    p.close()
    p.join()
    print(f"{round(end-start, 2)} seconds")


PicklingError: Can't pickle <class 'xsimlab.process.Backend'>: attribute lookup Backend on xsimlab.process failed

In [6]:
import xsimlab.process
import pickle

#backend = xsimlab.process.Backend  # Just the class

# Try to pickle it
pickle.dumps(model)


PicklingError: Can't pickle <class 'xsimlab.process.Backend'>: attribute lookup Backend on xsimlab.process failed

In [10]:
model_setup.xsimlab.

<method-wrapper '__str__' of SimlabAccessor object at 0x166670180>

In [19]:
#model.__dict__

In [17]:
model.Core

<Backend 'Core' (xsimlab process)>
Variables:
    solver_type     [in] solver type to use for model
    core           [out] model backend instance is stored here
    m              [out] math wrapper functions provided by solver
Simulation stages:
    initialize
    finalize

In [16]:
model.Core.__class__

xsimlab.process.Backend

In [13]:


# Extract the class from the model
backend_cls = model.processes['Core'].__class__

# Check where Python *thinks* it's from
print("Backend class module:", backend_cls.__module__)
print("Backend class name:", backend_cls.__name__)


AttributeError: 'Model' object has no attribute 'processes'

In [7]:
import xsimlab as xs

class _Backend:
    """... docstring ..."""

    solver_type = xs.variable(intent='in')
    core = xs.any_object()
    m = xs.any_object()

    def initialize(self):
        from .core import XSOCore
        self.core = XSOCore(self.solver_type)
        self.m = self.core.solver.MathFunctionWrappers

    def finalize(self):
        self.core.cleanup()

# Manually register the component with xsimlab
Backend = xs.process(_Backend)

# Patch the module so pickle knows where to look
Backend.__module__ = 'xso.backendcomps'
Backend.__name__ = 'Backend'

In [8]:
Backend.__class__.__mro__

(type, object)

In [1]:
from func_parallel_tests import Variable

In [2]:
import dill
dill.dumps(Variable)

/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class 'xso.component.Variable'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aoop/mambaforge/envs/XSO2025/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class 'xso.component.Variable'>: xso.component.Variable has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


PicklingError: Can't pickle <class 'xso.component.Variable'>: it's not found as xso.component.Variable